# Noise Power Spectral Density $S_N(f)$ Modeling

## Overview

The total noise PSD $S_N(f)$ is modeled as the sum of four statistically independent noise contributions:

$$S_{N}(f) = S_{RIN} \cdot |H_{ch}(f)|^2 + S_{shot}(f) + S_{th}(f) + S_{ADC}(f)$$

Each component is detailed below.



In [2]:
import numpy as np
# Physical constants

q = 1.602176634e-19 # Eletron charge (C)

## 1. Relative Intensity Noise (RIN)

RIN originates from the transmitter laser. Unlike other noise sources added at the receiver, RIN is added at the transmitter and is consequently filtered by the linear channel's transfer function $H_{ch}(f)$.

The RIN PSD is proportional to the square of the average transmitted optical power $\overline{P_{TX}^2}$:

$$S_{RIN} = k_{RIN} \cdot \overline{P_{TX}^2}$$

Where:

* $k_{RIN} = RIN_{coeff} / 2$ is a proportionality factor.


* $RIN_{coeff}$ is the RIN coefficient expressed in 1/Hz.

* $\overline{P_{TX}^2}$ is the average transmitted optical power squared.

In [3]:
def calc_S_RIN(P_TX_sq_avg, RIN_coeff_dB):
    """
    Calcula a PSD do RIN na fonte. 
    k_RIN = RIN_coeff_linear / 2.
    Retorna um escalar, pois o filtro do canal será aplicado na função principal.
    """
    # Converte de dB/Hz para escala linear
    RIN_coeff_lin = 10**(RIN_coeff_dB / 10)
    k_RIN = RIN_coeff_lin / 2
    return k_RIN * P_TX_sq_avg

## 2. Shot Noise

Shot noise is a quantum effect associated with the photodetection process at the receiver. Its variance scales with the received signal power. For this analytical frequency-resolved model, it is approximated using the average received optical power $P_{RX}$.

$$S_{shot}(f) = k_{shot} \cdot P_{RX}$$

Where $k_{shot}$ is a proportionality factor defined as:

$$k_{shot} = G^2 \cdot F \cdot q \cdot R^{-1}$$

* **$G$**: Photodetector gain (for PIN photodiodes, $G = 1$; for APDs, $G > 1$).


* **$F$**: Photodetector excess noise figure (for PIN, $F = 1$).


* **$q$**: Electron charge ($1.602 \times 10^{-19}$ C).


* **$R$**: Photodiode responsivity (A/W).

In [4]:
def calc_S_shot(P_RX, G, F, R, f_array):
    """
    Calcula a PSD do ruído de disparo (shot noise).
    Assumindo k_shot = G^2 * F * q / R.
    """
    k_shot = (G**2 * F * q) / R
    S_shot_val = k_shot * P_RX
    return np.full_like(f_array, S_shot_val)


## 3. Thermal Noise

Thermal noise is treated as additive white Gaussian noise (AWGN) and is assumed to be constant across the frequency spectrum. It is typically dominated by the internal noise generated by the Transimpedance Amplifier (TIA) at the receiver.

$$S_{th}(f) = \frac{N_0}{2}$$

Where $N_0$ is the equivalent noise power spectral density in $\text{W}^2/\text{Hz}$.

In [5]:
def calc_S_th(N_0, f_array):
    """
    Calcula a PSD do ruído térmico, assumido como constante na frequência.
    Baseado em S_th = N_0 / 2.
    """
    return np.full_like(f_array, N_0 / 2)

## 4. ADC Quantization Noise

The analog-to-digital conversion process at the receiver introduces quantization noise, which is dependent on the resolution of the ADC. The PSD of the quantization noise is modeled as:

$$S_{ADC}(f) = \frac{PAPR^{-2} \cdot \sigma_x^2 \cdot (2^{-2 \cdot ENOB})}{12 \cdot f_s / 2}$$

Where:

* **$PAPR$**: Peak-to-Average Power Ratio of the signal.


* **$ENOB$**: Effective Number of Bits of the oscilloscope/ADC.


* **$f_s$**: Sampling frequency.


*  **$\sigma_x^2$**: Power of the AC-coupled signal after quantization, calculated as $\sigma_x^2 = \int_{-f_s/2}^{f_s/2} S_x(f) df$.

In [6]:
def calc_S_ADC(PAPR, sigma_x_sq, ENOB, f_s, f_array):
    """
    Calcula a PSD do ruído de quantização do ADC.
    Baseado na Equação (9) do artigo.
    """
    S_ADC_val = ((PAPR**-2) * sigma_x_sq * (2**(-2 * ENOB))) / (12 * f_s / 2)
    return np.full_like(f_array, S_ADC_val)